In [ ]:
# import newkernelo as lib
import xllim as lib
import numpy as np
import time
import os.path
import json
import logging
logging.getLogger().setLevel(logging.INFO)

In [ ]:
# model =lib.FunctionalModel()
# help(lib.FunctionalModel)
# a = model.getDimensionY()
# print(model)

In [ ]:
model = lib.TestModel()
print(model.__doc__)
print(lib.ShkuratovModel.__doc__)
# print(lib.FunctionalModel.__doc__)
# print(lib.ExternalPythonModel.__doc__)
# print(lib.ExternalPythonModel.__init__.__doc__)
help(lib.TestModel)


## Test Model

In [ ]:
model = lib.TestModel()
e = np.exp(1)
x_test = np.zeros(4)
y_test = model.F(x_test)
true_result = np.array([4+2*e, 0.5, e, 3, 0.2, -0.5, -0.2-e, 2*e-1, -0.7])*0.5
print("y_test")
print(y_test)
print("true_result")
print(true_result)

In [ ]:
e = np.exp(1)
x = np.ones(4)

# y = np.zeros(9)
# t0 = time.time()
# for i in range(10000000): #10000000
#     model.F(x,y)
# tf = time.time() - t0
# print("Time F by reference: {}".format(tf))

# t0 = time.time()
# for i in range(10000000):
#     z = model.F(x)
# tf = time.time() - t0
# print("Time F by value: {}".format(tf))

# true_result = np.array([4+2*e, 0.5, e, 3, 0.2, -0.5, -0.2-e, 2*e-1, -0.7])*0.5
# print("true_result")
# print(true_result)

z = model.F(x)
print(z)
print(x)
print(x.shape)
model.toPhysic(x) # does nothing
print(x)
print(x.shape)
w = model.toPhysic(x) # return vec size-like : w.shape() = (5,1)
print(x)
print(w)
print(w.shape)
print("=========== From physic ==========")
print(x)
model.fromPhysic(x)
print(x)
y = model.fromPhysic(x)
print(x)
print(y)
z = model.fromPhysic(y)
print(x)
print(y)
print(z)

## Shkuratov

In [ ]:
print(os.getcwd())
with open('../dataRef/Shkuratov5p_data_ref.json', 'r') as f:
    data = json.load(f)

D = 50
row_size = D
col_size = 3

scalingCoeffs = [1.0,1.5,1.5,1.5,1.5]
offset = [0,0,0.2,0,0]


# # Create JSON file with geometries
geometries = np.empty((row_size,col_size))
var_geom = ["inc", "eme", "phi"]
for j in range(3):
    i=0
    for v in data[var_geom[j]]:
        geometries[i,j] = v # geometries.shape = (D,3)
        i+=1
# print(geometries)
print(geometries.shape)

# geom = {'geometries': [[geometries[j,:].tolist()] for j in range(3)]
# }
# with open('geometries_shkuratov.json', 'w') as fp:
#     json.dump(geom, fp)

## INTEGRATION au code C++
variant = "5p"
physicalModel = lib.ShkuratovModel(geometries, variant, scalingCoeffs, offset)


### TEST
N = 100
L = 5
variables = ["an", "mu1", "nu", "m", "mu2"]
photometries = np.empty((L,N))

# Read photometries
for l in range(L):
    for n in range(N):
        photometries[l,n] = (float(data[variables[l]][n]) - offset[l]) / scalingCoeffs[l]
        n+=1


# Read expected results
expected_results = np.empty((D,N))
n=0
for n in range(N):
    for d in range(D):
        expected_results[d,n] = float(data["y"][n][d])



# compute results from the model
# result = np.empty((D,))
assert_list = []
for n in range(N):
    result = physicalModel.F(photometries[:,n])
    assert_list.append(np.allclose(expected_results[:,n], result, rtol=1e-8))

print(result.shape)
print(assert_list)
print(False in assert_list)
print(True in assert_list)

print(expected_results[:10,n])
print(result[:10])


## Hapke

In [ ]:
with open('../dataRef/Hapke6p_geom70_data_ref.json') as json_file:
    data = json.load(json_file)
    expected_results = np.array(data["data_ref"]["synthetic_dataset"]['Y'])
    geometries = np.array(data["data_ref"]['geometries'])
    photometries = np.array(data["data_ref"]["synthetic_dataset"]['X'])
print(geometries.shape)
print(photometries.shape)
print(expected_results.shape)


hapkeModel = lib.HapkeModel(geometries, "2002", "six", 30.0,1,0)
print(hapkeModel.getDimensionY())
print(hapkeModel.getDimensionX())

# TEST
N = 10
assert_list = []
for n in range(N):
    result = hapkeModel.F(photometries[n])
    assert_list.append(np.allclose(expected_results[n], result, rtol=1e-8))

print(result.shape)
print(assert_list)
print(False in assert_list)
print(True in assert_list)

print(expected_results[n])
print(result)

In [ ]:
with open('../dataRef/Hapke4p_geom70_data_ref.json') as json_file:
    data = json.load(json_file)
    expected_results = np.array(data["data_ref"]["synthetic_dataset"]['Y'])
    geometries = np.array(data["data_ref"]['geometries'])
    photometries = np.array(data["data_ref"]["synthetic_dataset"]['X'])
print(geometries.shape)
print(photometries.shape)
print(expected_results.shape)


hapkeModel = lib.HapkeModel(geometries, "2002", "four", 30.0,1,0)
print(hapkeModel.getDimensionY())
print(hapkeModel.getDimensionX())

# TEST
N = 10
assert_list = []
for n in range(N):
    result = hapkeModel.F(photometries[n])
    assert_list.append(np.allclose(expected_results[n], result, rtol=1e-8))

print(result.shape)
print(assert_list)
print(False in assert_list)
print(True in assert_list)

print(expected_results[n,:10])
print(result[:10])

### NOTE : le expected_results[n][2] et result[2] sont différents.... mais les autres sont égaux. à voir...

In [ ]:
print(hapkeModel.getDimensionY())
print(hapkeModel.getDimensionX())
x = np.ones(hapkeModel.getDimensionX()) / 10

print(x)
print(x.shape)
hapkeModel.toPhysic(x) # does nothing
print(x)
print(x.shape)
w = hapkeModel.toPhysic(x) # return vec size-like : w.shape() = (5,1)
print(x)
print(w)
print(w.shape)
print("=========== From physic ==========")
print(x)
hapkeModel.fromPhysic(x)
print(x)
y = hapkeModel.fromPhysic(x)
print(x)
print(y)
z = hapkeModel.fromPhysic(y)
print(x)
print(y)
print(z)

## External model

In [ ]:
externalPythonModel = lib.ExternalPythonModel("ShkuratovModel5p", "ShkuratovModel5pPython", "../dataRef/")

print(externalPythonModel.getDimensionY())
print(externalPythonModel.getDimensionX())

In [ ]:
x = np.ones(externalPythonModel.getDimensionX())

print(x)
print(x.shape)
print("=========== To physic ==========")
externalPythonModel.toPhysic(x) # does nothing
print(x)
print(x.shape)
w = externalPythonModel.toPhysic(x) # return vec size-like : w.shape() = (5,1)
print(x)
print(w)
print(w.shape)
print("=========== From physic ==========")
externalPythonModel.fromPhysic(x)
print(x)
y = externalPythonModel.fromPhysic(x)
print(x)
print(y)
z = externalPythonModel.fromPhysic(y)
print(x)
print(y)
print(z)

In [ ]:
with open('../dataRef/Shkuratov5p_data_ref.json', 'r') as f:
    data = json.load(f)

D = 50
row_size = D
col_size = 3

scalingCoeffs = [1.0,1.5,1.5,1.5,1.5]
offset = [0,0,0.2,0,0]



### TEST
N = 3
L = 5
variables = ["an", "mu1", "nu", "m", "mu2"]
photometries = np.empty((L,N))

# Read photometries
for l in range(L):
    for n in range(N):
        photometries[l,n] = (float(data[variables[l]][n]) - offset[l]) / scalingCoeffs[l]
        n+=1


# Read expected results
expected_results = np.empty((D,N))
n=0
for n in range(N):
    for d in range(D):
        expected_results[d,n] = float(data["y"][n][d])


# compute results from the model
# result = np.empty((D,))
assert_list = []
for n in range(N):
    result = externalPythonModel.F(photometries[:,n])
    assert_list.append(np.allclose(expected_results[:,n], result, rtol=1e-8))

print(result.shape)
print(assert_list)
print(False in assert_list)
print(True in assert_list)

print(expected_results[:10,n])
print(result[:10])

## dataGeneration

### New FunctionalModel.dataGen()

In [ ]:
# Get JSC1 geometries from JSON file
with open("JSC1_BRDF.json", "r") as f:
    data = json.load(f)
geometries_JSC1 = np.array(data["JSC1_analogue"]["geometries"], dtype=float)

variant = "2002"
adapter = "four"
theta_bar_scaling = 30.0
b0 = 0
h = 0.1

model_hapke_4p = lib.HapkeModel(geometries_JSC1, variant, adapter, theta_bar_scaling, b0, h)

In [ ]:
t = time.time()
x_gen, y_gen = model_hapke_4p.genData(1_000_000, "sobol", np.ones(model.getDimensionY()), 1234)
print(time.time() - t)

In [ ]:
print(x_gen.shape)
print(y_gen.shape)

In [ ]:
x_gen, y_gen = model.genData(400, "sobol", 0.1, 1234)

In [ ]:
print(x_gen.shape)
print(y_gen.shape)

### Importance sampling

In [ ]:
K = 3
L = 4

weight = np.ones(K) * 1/K
# weight = np.array([0.2, 0.1, 0.1, 0.2, 0.4])
# mean = np.ones((4,5))*0.77
# mean[:,1] *= 3
mean = np.random.rand(L,K)
cube =  np.ones((L,L,K))*0.01
cube += np.random.rand(L,L,K) *0.1
for k in range(cube.shape[2]):
    cube[:,:,k] += np.eye(L) * 0.1
    cube[:,:,k] = np.dot(cube[:,:,k], cube[:,:,k].T) * 0.001

proposition_gmms = [(weight.T, mean, cube), (weight.T, mean*0.2, cube*0.2)] * 200
y = np.array(y_gen[:2]) # Note avec CARMA, on a l'erreur "this array cannot be borrow...." dans le cas où on met en argument un pointeur! Et donc la mémoire n'appartient pas à cette variable.
y = y_gen
y_err = y*0.001
covariance = np.ones(model.getDimensionY()) *0.003
N_0 = 10000
B = 500
J = 20

In [ ]:
print(cube[:,:,0].T)

In [ ]:
verbose = 1
results = model.importanceSampling(proposition_gmms, y, y_err, covariance, N_0, B, J, verbose)

In [ ]:
def compute_reconstruction_error(reconstruction, observation):
    return np.linalg.norm(observation - reconstruction) / np.linalg.norm(observation)

In [ ]:
import kernelo as ker

# Create "physical" model
physical_model = ker.TestModelConfig().create()

# Create StatModel
# covariances = np.random.uniform(0, 0.0001, physical_model.get_D_dimension())
stat_model = ker.GaussianStatModelConfig("sobol", physical_model, covariance, 123454).create()

mean_prop_laws = []
for k in range(len(proposition_gmms)):
    mean_prop_laws.append(
        ker.GaussianMixturePropositionConfig(proposition_gmms[k][0], proposition_gmms[k][1], proposition_gmms[k][2].T).create()
    )
sampler_imis = ker.ImisConfig(N_0, B, J, stat_model).create()
# sampler_imis = ker.ImportanceSamplingConfig(N_0, stat_model).create()
result_old_ker = []


In [ ]:
tic = time.time()
for i in range(len(mean_prop_laws)):
    result_old_ker.append(sampler_imis.execute(mean_prop_laws[i], y[i], y_err[i]))
tac = time.time()
time.sleep(1)
print("Time: {}".format(tac - tic))

In [ ]:
print("New version")
print("Old Kernelo version")
for i in range(2):
    print("\nN_obs = {}".format(i))
    print(results.predictions[:,i])
    print(result_old_ker[i].mean)
    print(results.predictions_variance[:,i])
    print(result_old_ker[i].covariance)

In [ ]:
print("New version:")
print(results.nb_effective_sample.shape)
print(results.nb_effective_sample)
print(results.effective_sample_size)
print(results.qn)
